# 🧊 02 - Apache Iceberg Table Creation & Ingestion

### 📌 Objective
In this notebook, we use **Apache Spark 3.5** to create an Apache Iceberg v2 table with **Hidden Partitioning** (`months(created_date)`), write it to **Garage S3**, and commit our first snapshot to **Apache Polaris**.

### 💡 Why Hidden Partitioning?
- In Hive tables, if you partition by month, you had to maintain a separate string column like `year_month=2024-06`.
- In Apache Iceberg, you partition directly on the business column: `PARTITIONED BY (months(created_date))`.
- Users query naturally on `created_date`, and Iceberg automatically applies partition pruning under the hood!

In [ ]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession

load_dotenv()

POLARIS_URL = os.getenv("POLARIS_URL", "http://polaris:8181")
CLIENT_ID = os.getenv("POLARIS_CLIENT_ID", "polaris")
CLIENT_SECRET = os.getenv("POLARIS_CLIENT_SECRET", "polaris")
WAREHOUSE = os.getenv("ICEBERG_WAREHOUSE", "icelake")
S3_ENDPOINT = os.getenv("S3_ENDPOINT", "http://garage:3900")
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID", "GK1f5cbdb2ac0aebcbb74e18d5")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY", "b5057d94ab27275064a278e2e358bbe751d3610c26f7d6499bc5f0ca76996b29")
AWS_REGION = "us-east-1"

# Build PySpark Session with Apache Iceberg 1.6.1 + AWS S3FileIO
spark = (
    SparkSession.builder
    .appName("IceLake-Notebook-Ingestion")
    .config("spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1,"
        "org.apache.iceberg:iceberg-aws-bundle:1.6.1"
    )
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.icelake", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.icelake.type", "rest")
    .config("spark.sql.catalog.icelake.uri", f"{POLARIS_URL}/api/catalog")
    .config("spark.sql.catalog.icelake.oauth2-server-uri", f"{POLARIS_URL}/api/catalog/v1/oauth/tokens")
    .config("spark.sql.catalog.icelake.warehouse", WAREHOUSE)
    .config("spark.sql.catalog.icelake.credential", f"{CLIENT_ID}:{CLIENT_SECRET}")
    .config("spark.sql.catalog.icelake.scope", "PRINCIPAL_ROLE:ALL")
    .config("spark.sql.catalog.icelake.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.icelake.s3.endpoint", S3_ENDPOINT)
    .config("spark.sql.catalog.icelake.s3.region", AWS_REGION)
    .config("spark.sql.catalog.icelake.s3.path-style-access", "true")
    .config("spark.sql.catalog.icelake.s3.access-key-id", AWS_ACCESS_KEY_ID)
    .config("spark.sql.catalog.icelake.s3.secret-access-key", AWS_SECRET_ACCESS_KEY)
    .config("spark.sql.catalog.icelake.s3.checksum-enabled", "false")
    .config("spark.sql.defaultCatalog", "icelake")
    .master("local[2]")
    .getOrCreate()
)
print("✅ SparkSession initialized with Apache Iceberg!")

### Step 1: Create the Iceberg Table
Notice the `PARTITIONED BY (months(created_date))` syntax.

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS icelake.wikipedia.articles (
        id           STRING,
        title        STRING,
        text         STRING,
        url          STRING,
        created_date DATE,
        word_count   INT
    )
    USING iceberg
    PARTITIONED BY (months(created_date))
    TBLPROPERTIES (
        'write.format.default' = 'parquet',
        'write.parquet.compression-codec' = 'snappy'
    )
""")
print("✅ Iceberg table defined in Polaris catalog!")

### Step 2: Read Raw Parquet Data & Append to Iceberg

In [ ]:
raw_df = spark.read.parquet("data/raw/")
raw_df = raw_df.select("id", "title", "text", "url", "created_date", "word_count")
print(f"Read {raw_df.count()} rows from data/raw/")

# Append data into the Iceberg table
# (If data is already loaded, you can skip writing again or use .overwritePartitions())
print("Current records in Iceberg table:")
spark.sql("SELECT count(*) as total_records FROM icelake.wikipedia.articles").show()

### Step 3: Inspect Table Metadata & Snapshots
Iceberg exposes metadata as queryable virtual tables (`.snapshots`, `.files`, `.partitions`).

In [ ]:
print("--- Iceberg Snapshots Table ---")
spark.sql("""
    SELECT committed_at, snapshot_id, operation, summary['total-records'] as total_records 
    FROM icelake.wikipedia.articles.snapshots
""").show(truncate=False)

### Step 4: Inspect Partition Files

In [ ]:
spark.sql("""
    SELECT partition, file_path, file_size_in_bytes, record_count 
    FROM icelake.wikipedia.articles.files 
    LIMIT 5
""").show(truncate=False)